# TASK DESCRIPTION

**Legend:**

Young Alex has a beloved BERT model that he carries everywhere on his trusty flash drive. One day, during an excursion along the River Styx, a few drops of water landed on the precious device, corrupting the model's weights.

Heartbroken, Alex rushed home to fix the neural network. After quick analysis, he discovered that only the token embeddings were damaged. The rest of the architecture, including the attention blocks and heads, remained intact.

Now he needs to restore the model's performance while leaving all other weights frozen. No changes to the attention mechanisms or other components are allowed.

**Task:**

Fix the broken vectors in the model's embedding matrix so as to improve the model's quality on a text sentiment analysis task.

**Restrictions:**

- You may not use any other transformer-based pretrained models or LLMs.
- You may not use any additional data.
- You may not fine-tune or pretrain the model.

When submitting, make a Quick Save of the notebook; otherwise the solution may be rejected.

You must solve this task on Kaggle, not Cloud.ru.


# DEPENDINGS

In [82]:
import hashlib
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

np.random.seed(21)
torch.manual_seed(21)

device = "cuda" if torch.cuda.is_available() else "cpu"

# LOAD DATASET

Upload kaggle.json file to colab.

In [83]:
# Local setup: the Kaggle archive is already stored beside this notebook.
# No kaggle.json, chmod, mv, or shell unzip is needed when running locally.
NOTEBOOK_DIR = Path(r"D:/projects/Supervised-Learning-Experiments/olympiads/competition_samples/raw/neoai-2025-sparse/5_Broken_BERT")
ZIP_PATH = NOTEBOOK_DIR / "neoai-2025-broken-bert.zip"
print(f"Notebook data directory: {NOTEBOOK_DIR}")
print(f"Archive present: {ZIP_PATH.exists()}")

Notebook data directory: D:\projects\Supervised-Learning-Experiments\olympiads\competition_samples\raw\neoai-2025-sparse\5_Broken_BERT
Archive present: True


In [84]:
# Optional Kaggle command kept as documentation only.
# If you need to refresh the archive manually, download competition data for:
# neoai-2025-broken-bert
print("Using local archive/files; skipping Kaggle CLI download.")

Using local archive/files; skipping Kaggle CLI download.


In [85]:
# Extract the required CSV files from the local archive if they are missing.
required_csvs = ["val_dataset.csv", "test.csv"]
missing = [name for name in required_csvs if not (NOTEBOOK_DIR / name).exists()]

if missing:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"Missing {missing} and archive not found: {ZIP_PATH}")
    with zipfile.ZipFile(ZIP_PATH) as archive:
        names = set(archive.namelist())
        for name in missing:
            if name not in names:
                raise FileNotFoundError(f"{name} not found inside {ZIP_PATH}")
            archive.extract(name, NOTEBOOK_DIR)
            print(f"Extracted {name}")
else:
    print("CSV files already present.")

CSV files already present.


In [86]:
candidate_dirs = [
    NOTEBOOK_DIR,
    Path.cwd(),
    Path.cwd() / "neoai-2025-broken-bert",
    Path("/kaggle/input/neoai-2025-broken-bert"),
    Path(r"E:/IOAI/kits/neoai-2025/broken-bert"),
]

for dataset_dir in candidate_dirs:
    val_data_path = dataset_dir / "val_dataset.csv"
    test_data_path = dataset_dir / "test.csv"
    if val_data_path.exists() and test_data_path.exists():
        break
else:
    searched = "\n".join(str(path) for path in candidate_dirs)
    raise FileNotFoundError(
        "Could not find val_dataset.csv and test.csv. "
        "Place both files in one of these folders:\n"
        f"{searched}"
    )

print(f"Using dataset directory: {dataset_dir}")
val_df = pd.read_csv(val_data_path)
test_df = pd.read_csv(test_data_path)
print("validation:", val_df.shape, list(val_df.columns))
print("test:", test_df.shape, list(test_df.columns))
print(val_df.head())

Using dataset directory: D:\projects\Supervised-Learning-Experiments\olympiads\competition_samples\raw\neoai-2025-sparse\5_Broken_BERT
validation: (2500, 3) ['id', 'text', 'labels']
test: (2499, 2) ['id', 'text']
   id                                               text    labels
0   0  simple recipe for creamy spaghetti with bacon,...   neutral
1   1                                        wow heather   neutral
2   2  @Djalfy I sound really Brummie lol but most of...  negative
3   3  the fact my room is so hot is making me feel sick  negative
4   4                       just came back from the mall   neutral


# LOAD TOKENIZER & MODEL

In [87]:
from torchinfo import summary

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("Ilseyar-kfu/broken_bert")
model = AutoModelForSequenceClassification.from_pretrained("Ilseyar-kfu/broken_bert")
for param in model.parameters():
    param.requires_grad_(False)
print(summary(model))
print("---")
print(type(tokenizer).__name__)
print(type(model).__name__)
print(model.config.id2label)
print("embedding shape:", tuple(model.bert.embeddings.word_embeddings.weight.shape))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4909.76it/s]

Layer (type:depth-idx)                                  Param #
BertForSequenceClassification                           --
├─BertModel: 1-1                                        --
│    └─BertEmbeddings: 2-1                              --
│    │    └─Embedding: 3-1                              (23,440,896)
│    │    └─Embedding: 3-2                              (393,216)
│    │    └─Embedding: 3-3                              (1,536)
│    │    └─LayerNorm: 3-4                              (1,536)
│    │    └─Dropout: 3-5                                --
│    └─BertEncoder: 2-2                                 --
│    │    └─ModuleList: 3-6                             (85,054,464)
│    └─BertPooler: 2-3                                  --
│    │    └─Linear: 3-7                                 (590,592)
│    │    └─Tanh: 3-8                                   --
├─Dropout: 1-2                                          --
├─Linear: 1-3                                           (2,307)
To

In [88]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [89]:
val_encodings = tokenizer(val_df["text"].to_list(), truncation=True, padding=True, max_length=256)
val_dataset = Dataset(val_encodings, val_df["labels"].to_list())

In [90]:
texts_2_score = val_df["text"].to_list() + test_df["text"].to_list()
print(texts_2_score)

['simple recipe for creamy spaghetti with bacon, corn, mushrooms and peppers  http://bit.ly/KtfBR', 'wow heather', '@Djalfy I sound really Brummie lol but most of all I just hate looking at myself!', 'the fact my room is so hot is making me feel sick', 'just came back from the mall', '@amy_p hahaha I wanted to eat that Chicken', 'coming home tomorrow with a car full of treasures', "@JujuDeRoussie come overhere then, the Dutch don't dub  might go in an hour #BringTaraBack", 'wishing you all a happy monday and a wonderful start to this week ! Make it a good one', 'baseball games whoo hooo  when to banquet yesterday at the hyhtt sooo fun', "@EllabellCullen3 I can't. I'm on my iPod and it doesn't have IM", 'I know what would make me really tired, put me to sleep and I would sleep good!  ...RICE!!! Lmao!', "getting ready to start my work week, it's so not TGIF for me!! It's monday", 'My boyfriend just broke his wrist, now he might need surgery  im so nervous', '2 more days before the big co

# MODEL CHANGES

### Solution: get average sub tokens' embeddings for each token with zero embedding.

Get tokens with zero indices

In [91]:
def print_dir(obj):
    for i in dir(obj):
        if i.startswith("_"):
            continue
        print(i)

In [92]:
from tqdm import tqdm

Get token indices with zero embeddings.

In [93]:
model_embeddings = model.bert.embeddings.word_embeddings.weight.detach().cpu()
zero_rows = (model_embeddings == 0).all(dim=1)
print(zero_rows.sum())
zero_indices = torch.nonzero(zero_rows).squeeze().numpy()
print(zero_indices)
print(model_embeddings[zero_indices].sum())

tensor(12208)
[    1     3     6 ... 30517 30518 30521]
tensor(0.)


Get tokenizer vocabulary:

In [94]:
token_to_ids = tokenizer.get_vocab()
ids_to_token = {v: k for k,v in token_to_ids.items()}
non_zero_ids_to_token = {v: k for k,v in token_to_ids.items() if v not in zero_indices}

Function to get all tokens from the tokenizer vocabulary that are included in the current token.

In [95]:
def get_sub_tokens(token, ids_to_token):
  sub_tokens = []
  for idx, sub_token in ids_to_token.items():
    if sub_token in token:
      sub_tokens.append(idx)
  return sub_tokens

For each token with zero embeddings, we find subtokens from the tokenizer vocabulary and replace the vectors of these tokens with the averaged embeddings of the subtokens:

In [96]:
for zero_index in tqdm(zero_indices):
  zero_token = ids_to_token[zero_index]
  sub_tokens = get_sub_tokens(zero_token, non_zero_ids_to_token)
  if len(sub_tokens) != 0:
    mean_embedding = model_embeddings[sub_tokens].mean(dim=0)
    model_embeddings[zero_index] = mean_embedding
  else:
    model_embeddings[zero_index] = torch.rand(1, model_embeddings.shape[1])

100%|██████████| 12208/12208 [00:16<00:00, 719.38it/s] 


In [97]:
model = AutoModelForSequenceClassification.from_pretrained("Ilseyar-kfu/broken_bert")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5088.52it/s]


Replace the model's embedding matrix with a new emedding matrix

In [98]:
non_zero = []
iters = 0
for i in model_embeddings:
    iters += 1
    if i.sum().item() != 0:
        non_zero.append(i)
non_zero = np.array(non_zero)
mean = torch.Tensor(non_zero.mean(axis=0)).requires_grad_(False)
# for i in model_embeddings:
#     if i.sum().item() == 0:
#         i = mean
model_embeddings.to(device)
with torch.no_grad():
    model.bert.embeddings.word_embeddings.weight.copy_(model_embeddings.to(device))

In [99]:
for i in model.parameters():
    if i.sum().item() == 0:
        print("-----")

# EVALUATION

In [100]:
from sklearn.metrics import f1_score
from numpy import argmax
from transformers import pipeline
import wandb
wandb.init(mode= "disabled");

In [101]:
from sklearn.metrics import classification_report

def evaluate_on_validation(model, tokenizer, df_val):
    label_2_dict = {'LABEL_0': 'neutral', "LABEL_1" : 'positive', "LABEL_2": 'negative'}
    classifier = pipeline("text-classification", model= model, tokenizer = tokenizer)
    answ = classifier.predict(list(df_val["text"]))
    answ = [label_2_dict[el["label"]] for el in answ]

    # print(f1_score(p.label_ids, preds, average='macro'))
    print(classification_report(df_val["labels"], answ))

In [102]:
evaluate_on_validation(model, tokenizer, val_df)

              precision    recall  f1-score   support

    negative       0.71      0.14      0.24       935
     neutral       0.34      0.86      0.49       759
    positive       0.59      0.27      0.37       806

    accuracy                           0.40      2500
   macro avg       0.54      0.43      0.37      2500
weighted avg       0.56      0.40      0.36      2500



# MODEL SCORING
When submitting the solution:

1. Make a Quick Save of the notebook; otherwise the solution may be rejected.
2. Add the notebook version in the submission comment.


In [103]:
import hashlib

def create_submission(model, tokenizer, df_test):
    label_2_dict = {'LABEL_0': 'neutral', "LABEL_1" : 'positive', "LABEL_2": 'negative'}
    classifier = pipeline("text-classification", model= model, tokenizer = tokenizer)
    answ = classifier.predict(list(df_test["text"]))
    answ = [label_2_dict[el["label"]] for el in answ]

    df = pd.DataFrame({"labels" : answ, "id": df_test['id']})
    hsh = hashlib.sha256(df.to_csv(index=False).encode('utf-8')).hexdigest()[:8]
    submit_path = f"submit_{hsh}.csv"
    print(f"SUBMIT_NAME: {submit_path}")
    print(df.head(10))
    df.to_csv(submit_path,index=False)

In [104]:
create_submission(model, tokenizer, test_df)

SUBMIT_NAME: submit_ce1fb555.csv
     labels    id
0  positive  5000
1   neutral  5001
2   neutral  5002
3   neutral  5003
4  positive  5004
5   neutral  5005
6   neutral  5006
7   neutral  5007
8  negative  5008
9   neutral  5009
